In [ ]:
import pandas as pd
import numpy as np
from functools import reduce

## Read and clean census data tables separately 

### Data Table DP02

In [ ]:
# By default 1st line in csv read as header of df
df = pd.read_csv('../data/raw/DP02.csv')   
print(df.shape)
df.head()

### In the csv file the 1st line is the variable names, 2nd line is selected variables with new names, 3nd line is metadata.

In [ ]:
# Keep columns where first row (iloc[0]) is not null
df = df.loc[:, df.iloc[0].notnull()]
print("Number of columns:", df.shape[1])
df.head()

In [ ]:
# Use 1st row as column names
df.columns = df.iloc[0]
# Remove first two rows (header + metadata)
df = df.iloc[2:]
# Reset index
df = df.reset_index(drop=True)
df.head()

In [ ]:
# Select columns 
print("Columns before:", df.columns)
df = df.drop(['total_pop_household', 'pop25'], axis=1)
print("Columns after:", df.columns)
print(df.info())

In [ ]:
# Convert data types
numeric_column_list = ['pct_high_school', 'pct_some_college',
       'pct_associate', 'pct_bachelor', 'pct_graduate', 'pct_same_house',
       'pct_diff_house', 'pct_diff_house_us', 'pct_same_county',
       'pct_diff_county', 'pct_diffCounty_sState', 'pct_diff_state',
       'pct_abroad', 'pct_native', 'pct_foreign_born']
for numeric_column in numeric_column_list:
    df[numeric_column] = df[numeric_column].astype(float)
print(df.info())

In [ ]:
# Check missing values
print(df.isnull().sum())

# Save the cleaned dataset
df.to_csv('../data/processed/DP02_clean.csv', index=False)

### Data Table DP03

In [ ]:
df = pd.read_csv('../data/raw/DP03.csv')   
print(df.shape)
# Keep columns where first row (iloc[0]) is not null
df = df.loc[:, df.iloc[0].notnull()]
print("Number of columns:", df.shape[1])
# Use 1st row as column names
df.columns = df.iloc[0]
# Remove first two rows (names + metadata)
df = df.iloc[2:]
# Reset index
df = df.reset_index(drop=True)
df.head()

In [ ]:
# Select columns 
print("Columns before:", df.columns)
df = df.drop(['pop16'], axis=1)
print("Columns after:", df.columns)
print(df.head())
print(df.info())

In [ ]:
# Convert data types
numeric_column_list = ['unemploy_rate', 'pct_below_poverty']
for numeric_column in numeric_column_list:
    df[numeric_column] = df[numeric_column].astype(float)
print(df.info())

In [ ]:
# Check missing values
print(df.isnull().sum())

# Save the cleaned dataset
df.to_csv('../data/processed/DP03_clean.csv', index=False)

### Data Table DP04

In [ ]:
df = pd.read_csv('../data/raw/DP04.csv')   
print(df.shape)
# Keep columns where first row (iloc[0]) is not null
df = df.loc[:, df.iloc[0].notnull()]
print("Number of columns:", df.shape[1])
# Use 1st row as column names
df.columns = df.iloc[0]
# Remove first two rows (names + metadata)
df = df.iloc[2:]
# Reset index
df = df.reset_index(drop=True)
df.head()

In [ ]:
# Select columns 
print("Columns before:", df.columns)
df = df.drop(['count_house_rent', 'count_house_owner', 
    'count_occupied_house', 'pct_owner_occupied',], axis=1)
print("Columns after:", df.columns)
print(df.dtypes) 

In [ ]:
# Check missing values
print(df.isnull().sum())

# Convert data types
df['pct_renter_occupied'] = df['pct_renter_occupied'].astype(float)

# Replace any non-numeric values with NaN
df = df.replace({'-': np.nan, 'None': np.nan, '': np.nan})
selected_columns = ['median_house_value', 'pct_houseValue300k', 'pct_houseValue500k', 'pct_houseValue1M', 'gross_rent_median']
df[selected_columns] = df[selected_columns].apply(pd.to_numeric, errors='coerce')
print(df.dtypes)

In [ ]:
# Save the cleaned dataset
df.to_csv('../data/processed/DP04_clean.csv', index=False)

### Data Table DP05

In [ ]:
df = pd.read_csv('../data/raw/DP05.csv')   
print(df.shape)
# Keep columns where first row (iloc[0]) is not null
df = df.loc[:, df.iloc[0].notnull()]
print("Number of columns:", df.shape[1])
# Use 1st row as column names
df.columns = df.iloc[0]
# Remove first two rows (names + metadata)
df = df.iloc[2:]
# Reset index
df = df.reset_index(drop=True)
df.head()

In [ ]:
# Select columns 
print("Columns before:", df.columns)
df = df.drop(['pop_male', 'pop_female', 'pop_65years', 'pct_female'], axis=1)
print("Columns after:", df.columns)
print(df.dtypes) 
print(df.head())

In [ ]:
# Check missing values
print(df.isnull().sum())

# Convert data types
num_col_list = ['median_age', 'pct_male', 'pct_under18', 'pct_age2024',
       'pct_age2534', 'pct_age3544', 'pct_age4554', 'pct_age5559',
       'pct_age6064', 'pct_under18', 'pct_over65']
for num_col in num_col_list:
    df[num_col] = df[num_col].astype(float)

print(df.dtypes)

In [ ]:
# Save the cleaned dataset
df.to_csv('../data/processed/DP05_clean.csv', index=False)

## Combined Data in One Table

In [ ]:
files = ['DP03_clean.csv', 'DP04_clean.csv', 'DP05_clean.csv'] 
df_list = [pd.read_csv('../data/processed/DP02_clean.csv')]
for file in files:
    df = pd.read_csv(f'../data/processed/{file}') 
    df = df.drop(columns=['census_tract'])
    df_list.append(df)

merge_key = 'geo_id'  
for i, df in enumerate(df_list):
    if merge_key not in df.columns:
        print(f"Warning: '{merge_key}' not found in file {files[i]}")
# merge data tables 
merge_key = 'geo_id'
df_merged = reduce(lambda left, right: pd.merge(left, right, on=merge_key, how='inner'), df_list)
print(f"Merged DataFrame shape: {df_merged.shape}")
df_merged.to_csv('../data/processed/combined_data.csv', index=False)

## Split Data 

In [23]:
df = pd.read_csv("../data/processed/combined_data.csv")
df.head()

,geo_id,census_tract,pct_high_school,pct_some_college,pct_associate,pct_bachelor,pct_graduate,pct_same_house,pct_diff_house,pct_diff_house_us,...,median_age,pct_male,pct_age2024,pct_age2534,pct_age3544,pct_age4554,pct_age5559,pct_age6064,pct_under18,pct_over65
0,1400000US16055000101,Census Tract 1.01; Kootenai County; Idaho,26.7,22.4,16.6,21.6,9.3,89.7,10.3,10.3,...,54.0,52.3,2.4,2.8,7.2,21.1,12.9,9.3,16.4,24.5
1,1400000US16055000102,Census Tract 1.02; Kootenai County; Idaho,32.1,32.1,13.2,12.7,6.0,88.5,11.5,11.5,...,47.3,49.8,5.0,8.5,12.5,12.2,9.8,12.1,20.1,18.4
2,1400000US16055000201,Census Tract 2.01; Kootenai County; Idaho,27.2,31.9,8.4,19.3,9.3,84.4,15.6,14.7,...,37.2,46.9,7.7,14.5,11.6,14.2,6.3,4.3,23.2,16.7
3,1400000US16055000202,Census Tract 2.02; Kootenai County; Idaho,23.7,21.7,17.7,18.0,13.9,91.5,8.5,8.5,...,54.3,49.8,2.3,4.9,8.3,17.8,11.8,12.9,18.9,21.0
4,1400000US16055000203,Census Tract 2.03; Kootenai County; Idaho,39.1,21.9,6.3,20.4,3.4,95.0,5.0,5.0,...,50.4,50.6,7.4,8.5,7.5,14.8,9.4,11.2,17.5,20.9


In [24]:
# Coeur d'Alene area (Kootenai County, Idaho)
df_cda = df[df["census_tract"].str.contains("Kootenai", na=False)]

# Spokane area (Spokane County, Washington)
df_spokane = df[df["census_tract"].str.contains("Spokane", na=False)]

In [25]:
print("CDA rows:", len(df_cda))
print("Spokane rows:", len(df_spokane))

CDA rows: 39
Spokane rows: 130


In [26]:
df_cda.to_csv("../data/processed/cda_data.csv", index=False)
df_spokane.to_csv("../data/processed/spokane_data.csv", index=False)